这次我们做一个完整的 PyTorch RNN 文本情感分类项目：输入一段英文电影评论，模型判断它是正面评价还是负面评价。

这个项目会把你想学习的三个核心知识点串起来：
1. Token ID：把文本中的单词转换成数字编号。
2. Embedding：把离散的 Token ID 转换成稠密向量。
3. Padding：把不同长度的文本补齐成相同长度，以便组成 Batch。
另外，我会加入 pack_padded_sequence，让 RNN 在计算时忽略补齐部分，避免 Padding 影响有效文本的最终隐藏状态。
一、项目选择：IMDb 电影评论情感分类



IMDb Movie Reviews
经典二分类数据集
50,000 条英文电影评论，正面与负面评论各 25,000 条。

适合学习 NLP、Tokenization、Embedding、RNN、LSTM 和文本分类。


数据集可以从以下地址获取：
- [Hugging Face：IMDb 数据集](https://huggingface.co/datasets/stanfordnlp/imdb)
- [Stanford 官方 IMDb 数据集页面](https://huggingface.co/docs/datasets/loading)
- [Hugging Face Datasets 加载文档](https://huggingface.co/docs/datasets/loading)

建议使用 Hugging Face datasets 库自动下载，不需要自己手动下载和解压数据集。

In [1]:
import os
import csv
# 安装 hugging face 库: uv add datasets
from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim


In [3]:
# 1、获取数据

import os
import csv
from datasets import load_dataset

# 创建数据目录
os.makedirs("../../data/dl/imdb/raw", exist_ok=True)

# 从 Hugging Face 下载 IMDb
dataset = load_dataset("stanfordnlp/imdb")

# 分别保存训练集和测试集
for split in ["train", "test"]:
    output_path = f"../../data/dl/imdb/raw/{split}.csv"

    with open(output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["text", "label"]
        )
        writer.writeheader()

        for item in dataset[split]:
            writer.writerow({
                "text": item["text"],
                "label": item["label"]
            })

    print(f"{split} 数据集保存成功：{output_path}")
    print(f"数据量：{len(dataset[split])}")

print("数据集下载完成！")


train 数据集保存成功：../../data/dl/imdb/data/raw/train.csv
数据量：25000
test 数据集保存成功：../../data/dl/imdb/data/raw/test.csv
数据量：25000
数据集下载完成！


这次我们做一个完整的 PyTorch RNN 文本情感分类项目：输入一段英文电影评论，模型判断它是正面评价还是负面评价。

这个项目会把你想学习的三个核心知识点串起来：

1. Token ID：把文本中的单词转换成数字编号。
2. Embedding：把离散的 Token ID 转换成稠密向量。
3. Padding：把不同长度的文本补齐成相同长度，以便组成 Batch。

另外，我会加入 `pack_padded_sequence`，让 RNN 在计算时忽略补齐部分，避免 Padding 影响有效文本的最终隐藏状态。

## 一、项目选择：IMDb 电影评论情感分类

[JAI | Free Full-Text | Opinion Mining on Movie Reviews Based on Deep Learning Models](https://images.openai.com/static-rsc-4/_Y_U0xM9m8AQoPCpCh6xLCG8s6ooEAMXBa0Stv4Q_GVCw3yU_bXHZR9DZGs9JmQXFtZJWsCTOc2Ts6JiVi1Oq3kvyFJfnRhsjqDWuBtW41Ewbyuf1v-AnyjHVE-DC3aKJJgAPPsndREYM7FJUXvxEuxD1YLPq6qyLSgzUfELiS8?purpose=inline)

IMDb Movie Reviews

经典二分类数据集

50,000 条英文电影评论，正面与负面评论各 25,000 条。

适合学习 NLP、Tokenization、Embedding、RNN、LSTM 和文本分类。

数据集可以从以下地址获取：

- [Hugging Face：IMDb 数据集](https://huggingface.co/datasets/stanfordnlp/imdb?utm_source=chatgpt.com)
- [Stanford 官方 IMDb 数据集页面](https://ai.stanford.edu/~amaas/data/sentiment/?utm_source=chatgpt.com)
- [Hugging Face Datasets 加载文档](https://huggingface.co/docs/datasets/loading?utm_source=chatgpt.com)

我建议使用 Hugging Face `datasets` 库自动下载，不需要自己手动下载和解压数据集。

## 二、项目目录结构

这次按照相对标准的 PyTorch 工程方式组织，不把所有代码堆在一个文件里。

```
rnn_text_classification/
│
├── data/
│   └── raw/
│       ├── train.csv             # 训练数据
│       └── test.csv              # 测试数据
│
├── models/
│   └── best_rnn.pth              # 最佳模型及词表
│
├── src/
│   └── text_classifier/
│       ├── __init__.py
│       ├── dataset.py            # Tokenize、词表、Dataset、Padding
│       └── model.py              # Embedding + RNN + Linear
│
├── scripts/
│   ├── download_data.py          # 下载数据集
│   ├── train.py                  # 训练模型
│   └── predict.py                # 评估和预测
│
└── requirements.txt
```

## 三、安装依赖

`requirements.txt`：

```
torch
numpy
pandas
datasets
scikit-learn
matplotlib
tqdm
```

在项目根目录执行：

```
# 建议先创建虚拟环境
python -m venv .venv

# macOS / Linux
source .venv/bin/activate

# Windows
# .venv\Scripts\activate

# 安装依赖
pip install -r requirements.txt
```

## 四、下载 IMDb 数据集

创建 `scripts/download_data.py`：

```
import osimport csvfrom datasets import load_dataset# 创建数据目录os.makedirs("data/raw", exist_ok=True)# 从 Hugging Face 下载 IMDbdataset = load_dataset("stanfordnlp/imdb")# 分别保存训练集和测试集for split in ["train", "test"]:    output_path = f"data/raw/{split}.csv"    with open(output_path, "w", encoding="utf-8", newline="") as f:        writer = csv.DictWriter(            f,            fieldnames=["text", "label"]        )        writer.writeheader()        for item in dataset[split]:            writer.writerow({                "text": item["text"],                "label": item["label"]            })    print(f"{split} 数据集保存成功：{output_path}")    print(f"数据量：{len(dataset[split])}")print("数据集下载完成！")
```

执行：

```
python scripts/download_data.py
```

IMDb 原始训练集有 25,000 条，测试集有 25,000 条，另外还有 50,000 条无标签数据。这里暂时只使用有标签的训练集和测试集。标签 `0` 表示负面，`1` 表示正面。

[image](https://www.google.com/s2/favicons?domain=https://huggingface.co\&sz=32)

Datasets at Hugging Face+1



## 五、数据预处理：Token ID、Embedding 前的数据准备、Padding

这是本项目最重要的部分之一。

创建 `src/text_classifier/dataset.py`。

### 1. 完整代码

```
import reimport csvfrom collections import Counterimport torchfrom torch.utils.data import Datasetfrom torch.nn.utils.rnn import pad_sequence# 特殊 TokenPAD_TOKEN = "<PAD>"UNK_TOKEN = "<UNK>"PAD_ID = 0UNK_ID = 1# 1. Tokenization：文本切分成 Tokendef tokenize(text):    """    将英文文本转换为单词列表。    """    text = text.lower()    tokens = re.findall(r"\b\w+\b|[^\w\s]", text)    return tokens# 2. 构建词表def build_vocab(texts, max_vocab_size=30000, min_freq=2):    """    将单词映射为唯一的整数 ID。    """    counter = Counter()    for text in texts:        tokens = tokenize(text)        counter.update(tokens)    # 预留 PAD 和 UNK    vocab = {        PAD_TOKEN: PAD_ID,        UNK_TOKEN: UNK_ID    }    # 按词频从高到低构建词表    for word, freq in counter.most_common():        if freq < min_freq:            continue        if word in vocab:            continue        if len(vocab) >= max_vocab_size:            break        vocab[word] = len(vocab)    return vocab
```

### 2. Token ID 到底是什么？

例如，原始评论：

```
This movie is very good
```

第一步：Tokenization。

```
tokens = [    "this",    "movie",    "is",    "very",    "good"]
```

第二步：根据词表转换成数字。

假设词表中存在以下映射：

```
vocab = {    "<PAD>": 0,    "<UNK>": 1,    "this": 2,    "movie": 3,    "is": 4,    "very": 5,    "good": 6}
```

那么文本就变成：

```
token_ids = [2, 3, 4, 5, 6]
```

这就是 Token ID。

注意：Token ID 只是单词的编号，并不代表单词的语义，也不代表数字越大语义越强。

### 3. Padding 是怎么工作的？

假设一个 Batch 中有三条评论：

```
[    [2, 3, 4, 5],    [6, 7],    [8, 9, 10]]
```

三条文本长度分别是：

```
4、2、3
```

RNN 训练时通常需要组成规则的二维批次张量，因此使用 `pad_sequence` 补齐：

```
[    [2, 3, 4, 5],    [6, 7, 0, 0],    [8, 9, 10, 0]]
```

得到：

Padding 后的 Token ID

## 2

## 3

## 4

## 5

## 6

## 7

## 0

## 0

## 8

## 9

## 10

## 0

真实 Token

Padding Token（0）

最终 Tensor Shape：[3, 4]，即 Batch Size = 3，序列长度 = 4。

## 六、定义 RNN 文本分类模型

创建 `src/text_classifier/model.py`：

```
import torchimport torch.nn as nnfrom torch.nn.utils.rnn import pack_padded_sequenceclass RNNTextClassifier(nn.Module):    def __init__(        self,        vocab_size,        embedding_dim=128,        hidden_size=128,        num_layers=1,        num_classes=2,        dropout=0.2    ):        super().__init__()        # 1. Embedding 层        self.embedding = nn.Embedding(            num_embeddings=vocab_size,            embedding_dim=embedding_dim,            padding_idx=0        )        # 2. RNN 层        self.rnn = nn.RNN(            input_size=embedding_dim,            hidden_size=hidden_size,            num_layers=num_layers,            batch_first=True,            nonlinearity="tanh",            dropout=dropout if num_layers > 1 else 0        )
```

### 这里重点理解 Embedding

```
self.embedding = nn.Embedding(    num_embeddings=vocab_size,    embedding_dim=128,    padding_idx=0)
```

假设：

```
vocab_size = 30000embedding_dim = 128
```

那么 Embedding 可以理解成一个可训练的矩阵：

\\[ E\in \mathbb{R}^{30000\times128} \\]

每个 Token ID 对应矩阵中的一行向量。

例如：

```
token_ids = torch.tensor([    [2, 3, 4, 0],    [5, 6, 0, 0]])
```

输入 Embedding：

```
embeddings = self.embedding(token_ids)
```

输出 Shape：

```
[2, 4, 128]
```

含义：

- `2`：Batch Size。
- `4`：每条文本补齐后的序列长度。
- `128`：每个 Token 对应的 Embedding 维度。

`padding_idx=0` 表示编号为 0 的 Padding Token 不参与 Embedding 权重更新，通常保持为零向量。

### 为什么还要使用 pack_padded_sequence？

Padding 解决的是张量形状不一致的问题，但它不会自动让 RNN 忽略 Padding。

例如：

```
[2, 3, 4, 5]
[6, 7, 0, 0]
```

如果直接输入普通 RNN，第二条文本后面的两个 0 也会进入循环计算。

`pack_padded_sequence` 会根据真实长度告诉 RNN：每条序列实际有多少个有效 Token。

因此，这个项目同时使用：

- `pad_sequence`：让一个 Batch 内的文本具有相同长度。
- `pack_padded_sequence`：让 RNN 忽略 Padding 对应的时间步。

这是变长文本 RNN 项目中非常重要的组合。

## 七、训练模型

创建 `scripts/train.py`：

```
import osimport sysimport jsonimport torchimport torch.nn as nnfrom torch.utils.data import DataLoader, Subsetfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import accuracy_scorefrom tqdm import tqdm# 让脚本能够导入 src 中的模块ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))sys.path.insert(0, os.path.join(ROOT_DIR, "src"))from text_classifier.dataset import (    load_csv,    build_vocab,    TextDataset,    collate_fn)from text_classifier.model import RNNTextClassifier# =========================# 1. 基础配置# =========================BATCH_SIZE = 64EPOCHS = 10LEARNING_RATE = 0.001MAX_VOCAB_SIZE = 30000MIN_FREQ = 2MAX_LEN = 300
```

### 训练过程中发生了什么？

每个 Batch 的完整流程：

```
原始文本
   │
   ▼
Tokenization
   │
   ▼
Token ID
   │
   ▼
Padding
   │
   ▼
Embedding
   │
   ▼
pack_padded_sequence
   │
   ▼
RNN
   │
   ▼
最后一层隐藏状态 h_n
   │
   ▼
Linear 分类头
   │
   ▼
Logits [batch_size, 2]
   │
   ▼
CrossEntropyLoss
   │
   ▼
Backward
   │
   ▼
Optimizer.step()
```

这里有几个值得注意的地方：

- `Embedding` 的权重会随着训练不断更新，因此模型能够学习单词的向量表示。
- `CrossEntropyLoss` 接收原始 logits，不需要提前执行 Softmax。
- `argmax(dim=1)` 取得预测类别。
- `clip_grad_norm_` 是为了缓解 RNN 训练时可能出现的梯度爆炸。
- 验证阶段使用 `model.eval()` 和 `torch.inference_mode()`，不会更新模型参数。

## 八、模型测试和实际文本预测

创建 `scripts/predict.py`：

```
import osimport sysimport torchfrom torch.utils.data import DataLoaderfrom sklearn.metrics import accuracy_score, classification_reportROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))sys.path.insert(0, os.path.join(ROOT_DIR, "src"))from text_classifier.dataset import (    load_csv,    TextDataset,    collate_fn,    encode_text)from text_classifier.model import RNNTextClassifier# =========================# 1. 设备# =========================if torch.cuda.is_available():    device = torch.device("cuda")elif torch.backends.mps.is_available():    device = torch.device("mps")else:    device = torch.device("cpu")# =========================# 2. 加载模型# =========================checkpoint_path = os.path.join(
```

执行：

```
# 下载数据
python scripts/download_data.py

# 训练模型
python scripts/train.py

# 测试模型并预测新文本
python scripts/predict.py
```

注意：第一次下载数据需要网络连接；如果 Hugging Face 访问较慢，可以先在可访问的网络环境中下载，再将 CSV 文件复制到 `data/raw/`。

## 九、关键 Tensor Shape 总结

这是整个项目最值得掌握的部分。假设：

```
batch_size = 4seq_len = 10embedding_dim = 128hidden_size = 128num_layers = 1num_classes = 2
```

那么数据经过模型时的 Shape 如下：

| 阶段              | Tensor       | Shape             |
| --------------- | ------------ | ----------------- |
| 原始 Token ID     | `token_ids`  | `[4, 10]`         |
| 记录真实长度          | `lengths`    | `[4]`             |
| Embedding       | `embeddings` | `[4, 10, 128]`    |
| Packed Sequence | `packed`     | PackedSequence 对象 |
| RNN 最终隐藏状态      | `h_n`        | `[1, 4, 128]`     |
| 取最后一层           | `h_n[-1]`    | `[4, 128]`        |
| 分类头输出           | `logits`     | `[4, 2]`          |
| 真实标签            | `labels`     | `[4]`             |

其中：

\\[ \text{Token ID}\xrightarrow{\text{Embedding}} \text{Word Embedding}\xrightarrow{\text{RNN}} \text{Hidden State}\xrightarrow{\text{Linear}} \text{Logits} \\]

## 十、建议你重点做的 5 个实验

完成上面的项目后，不要只满足于运行成功。建议通过以下实验理解模型内部机制。

| 实验   | 修改内容                          | 想要理解的问题                  |
| ---- | ----------------------------- | ------------------------ |
| 实验 1 | 将 `padding_idx=0` 去掉          | Padding Embedding 有什么区别？ |
| 实验 2 | 去掉 `pack_padded_sequence`     | Padding 为什么会影响 RNN？      |
| 实验 3 | `embedding_dim` 改成 64、128、256 | Embedding 维度有什么影响？       |
| 实验 4 | 将 `nn.RNN` 替换成 `nn.LSTM`      | LSTM 与 RNN 的效果有什么差异？     |
| 实验 5 | 将 `hidden_size` 改成 64、128、256 | 隐藏层维度如何影响模型？             |

尤其是实验 2：你可以对比使用普通 RNN 和 PackedSequence 时的验证集准确率，观察 Padding 对模型的影响。

## 十一、这个项目你应该真正掌握的知识

项目知识点检查清单

已掌握 0/12

0%

理解 Tokenization：文本如何切分成 Token

理解 Token ID：单词如何映射成整数

理解 Vocabulary：词表如何构建

理解 UNK：词表外的单词如何处理

理解 Embedding：整数如何映射成向量

理解 padding_idx：为什么需要 PAD Token

理解 pad_sequence：如何补齐变长文本

理解 lengths：如何记录真实序列长度

理解 pack_padded_sequence：如何跳过 Padding

理解 RNN：如何提取文本序列特征

理解 CrossEntropyLoss：如何进行文本分类

理解模型保存：为什么要同时保存词表和模型参数

最后强调一个工程细节： 训练时构建的词表必须在预测时继续使用，不能重新构建。否则同一个 Token 可能对应不同的 ID，模型就无法正确理解输入。所以上面的 `best_rnn.pth` 同时保存了 `vocab`、模型参数和模型配置。

这个 IMDb 项目是一个教学基线，不要把它的准确率当成固定指标；实际结果会受词表大小、序列长度、训练轮数、设备和随机初始化等因素影响。

下一步可以继续把这个项目改造成 LSTM 文本分类项目，对比 RNN 与 LSTM 在相同数据集、相同训练条件下的效果。

要不要明天继续做这个 RNN 与 LSTM 的对比实验？

📚明天进行 RNN 与 LSTM 对比实验轻点设置